# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** Entities are referenced by their `@id` fields.

In [ ]:
# List all record set @ids and summarize their fields
record_set_objects = list(dataset.record_sets.values())
print(f"Found {len(record_set_objects)} record sets:\n")
for rs in record_set_objects:
    print(f"RecordSet: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      Data type: {field.data_type}")
    print('-'*40)

# Show the first few records (example: first RecordSet)
if record_set_objects:
    first_rs_id = record_set_objects[0].id
    print(f"\nPreview of first 2 records in RecordSet {first_rs_id}:")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into DataFrames
record_sets_ids = [rs.id for rs in dataset.record_sets.values()]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if not dataframes[record_set_id].empty:
        print(f"Columns for RecordSet {record_set_id}: {dataframes[record_set_id].columns.tolist()}")

# Choose the first available DataFrame for exploration
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All entity references are by their `@id`.

In [ ]:
# Select a numeric field for analysis by its @id
if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]
    
    # Try to auto-detect a numeric field (float or int)
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # reference by field @id
        print(f"Using numeric field (@id): {numeric_field_id}\n")
        threshold = df[numeric_field_id].median()  # example threshold: median
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a string/categorical field (for groupby), by its @id
        object_cols = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = None
        for c in object_cols:
            # Exclude '@id', 'id', or description columns
            if (not c.lower().endswith('description')) and (c.lower() != '@id'):
                group_field_id = c
                break
        if group_field_id:
            print(f"\nGrouping by field (@id): {group_field_id}\n")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization for filtered and normalized numeric field
if main_record_set_id and not dataframes[main_record_set_id].empty and 'filtered_df' in locals():
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (>median)")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping field exists, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8, 4))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**
- The dataset was loaded, inspected for record sets/fields (all referenced by `@id`), and core numeric columns were filtered and normalized for analysis.
- Data exploration and visualization enable further biological or clinical interpretation of extracellular vesicle protein data.
- For advanced analysis, consult full field and record set documentation and use `mlcroissant` for robust, reproducible data loading.